# Step 2: Train YOLO26 on Swimming Pools

Trains YOLO26 variants (n/s/m/l) via transfer learning on the manually-cleaned Roboflow export. Reports every parameter the brief requires (optimizer, LR, epochs, imgsz, batch, augmentations, scheduler, hardware) and metrics (mAP50, mAP50-95, Precision, Recall, parameter count).

**Runtime:** Colab → Runtime → Change runtime type → **A100 GPU** (40 GB VRAM target). T4 / L4 also work without changes (yolo26l batch will be auto-OOM on T4 at the new batch=16; revert l-variant to 4 if downgrading).

**Pipeline:**
1. Mount Drive, extract Roboflow YOLO export
2. Train n / s / m / l with aerial-tuned hyperparameters
3. Comparison table + plots
4. Best-model test eval (with and without Test-Time Augmentation)
5. Failure-case dump (FPs and FNs from test set) for the mandatory write-up section
6. Bbox-size distribution (for the small-object discussion)
7. Augmentation ablation: yolo26n × 5 progressively-richer configurations
8. Hyperparameter tuning: Ultralytics genetic-algorithm tuner on yolo26n, then retrain with best HPs
9. Polygon → OBB label conversion (no re-annotation needed for Step 4)
10. Bundle artifacts to Drive


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — switch runtime to GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB  |  torch {torch.__version__}')

In [ ]:
%pip install -q 'ultralytics>=8.4.0' supervision pandas matplotlib opencv-python pyyaml
!yolo settings sync=False
import ultralytics; ultralytics.checks()

## Load dataset from Drive

Upload the Roboflow YOLO export zip (`Object Detection.v1i.yolov8.zip`) to your Drive at `MyDrive/IE/IndividualAssignmentMBD2026/`. The cell below mounts Drive, extracts the zip into `/content/dataset/`, and rewrites `data.yaml` paths to absolute.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, pathlib, shutil
ZIP_PATH = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026/Object Detection.v1i.yolov8.zip'
assert pathlib.Path(ZIP_PATH).exists(), f'Drive input missing at {ZIP_PATH}. Confirm Drive is mounted and the Roboflow YOLO zip is uploaded.'
DATA_DIR = pathlib.Path('/content/dataset')
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)

import yaml
yml = yaml.safe_load(open(DATA_DIR / 'data.yaml'))
yml['train'] = str(DATA_DIR / 'train' / 'images')
yml['val']   = str(DATA_DIR / 'valid' / 'images')
yml['test']  = str(DATA_DIR / 'test'  / 'images')
yaml.safe_dump(yml, open(DATA_DIR / 'data.yaml', 'w'))
DATA_YAML = str(DATA_DIR / 'data.yaml')
print('Data YAML:', DATA_YAML)
print(open(DATA_YAML).read())

for split in ['train', 'valid', 'test']:
    imgs = list((DATA_DIR / split / 'images').iterdir())
    print(f'{split}: {len(imgs)} images')

## Training configuration

All hyperparameters in one place so the write-up section can quote them verbatim.

| Parameter | Value | Justification |
|---|---|---|
| Optimizer | `AdamW` (explicit) | Set explicitly so `lr0`/`momentum` actually take effect. `optimizer='auto'` silently overrides them, so the hparam table would not match training. |
| Initial LR (`lr0`) | 0.005 | Half default — first run showed larger variants (s/m) early-stopped at epoch ~10 on 202 train images, classic overfit signature |
| LR scheduler | Cosine (`cos_lr=True`) | Smoother decay than linear; standard for fine-tuning |
| Epochs | 100 | Gives larger variants room to converge; early-stop via patience |
| Patience | 30 | Generous — first run with patience=10 fired far too early on yolo26s (112s of training) |
| Image size | 640×640 | Matches Roboflow stretched export. Source images are ~300×300 native, so higher imgsz would just upsample |
| Batch size | 16/16/8/16 (n/s/m/l) | A100 (40 GB); yolo26l bumped from 4 (T4 limit) on 2026-05-26 GPU upgrade. `cache='ram'` enabled (dataset ~200 MB at imgsz=640) |
| `close_mosaic` | 10 | Ultralytics best practice: disable mosaic for last 10 epochs, lets model converge on natural-looking images |
| Random seed | 0, `deterministic=True` | Reproducibility |
| **Augmentations** | | |
| `mosaic` | 1.0 | 4-image stitching; standard YOLO augmentation |
| `mixup` | 0.05 | Mild image blending; helps regularize on small datasets |
| `hsv_h/s/v` | 0.015/0.7/0.4 | Defaults; pool color varies with water clarity, time-of-day, depth |
| `fliplr` | 0.5 | Horizontal flip; pools are mirror-symmetric in aerial view |
| `flipud` | 0.5 | Vertical flip; aerial imagery has no canonical 'up' direction (added based on aerial-specific reasoning) |
| `degrees` | 180 | Full rotation; aerial pools can be at any orientation (added based on aerial-specific reasoning) |
| `translate` | 0.1 | Standard; pools can be at any image position |
| `scale` | 0.5 | Standard; handles varying altitudes |

In [ ]:
BASE_CFG = dict(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=30,
    optimizer='AdamW',           # explicit (was 'auto'): 'auto' silently overrides lr0/momentum, breaking the hparam-table contract
    lr0=0.005,
    cos_lr=True,
    close_mosaic=10,
    seed=0,
    deterministic=True,
    cache='ram',
    plots=True,
    verbose=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    degrees=180,
    mosaic=1.0, mixup=0.05,
    amp=True,  # explicit AMP for A100 (Ultralytics default is also True; documenting in hparam table)
    exist_ok=True,  # overwrite same-named runs/<name>/ on re-entry; prevents auto-incrementing dir names
)
VARIANT_BATCH = {'yolo26n': 16, 'yolo26s': 16, 'yolo26m': 8, 'yolo26l': 16}  # A100: l bumped 4 -> 16 (was T4-limited)
print('BASE_CFG:')
for k, v in BASE_CFG.items(): print(f'  {k:14s} = {v}')
print('\nPer-variant batch (A100):')
for k, v in VARIANT_BATCH.items(): print(f'  {k}: batch={v}')

## Training utility

`train_and_eval` runs one full training and validation cycle for a variant, captures the metrics the brief requires, frees GPU memory between runs.

In [ ]:
from ultralytics import YOLO
import time, gc, pathlib

results_summary = {}

def _free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print(f'  GPU mem: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated / {torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

def train_and_eval(variant: str, cfg=None, name_suffix='', store=True):
    """Train one YOLO variant, return val metrics. cfg defaults to BASE_CFG with per-variant batch."""
    key = variant + name_suffix
    if cfg is None:
        cfg = dict(BASE_CFG)
        cfg['batch'] = VARIANT_BATCH.get(variant, 16)
    print(f'\n{"="*60}\nTraining {key} (batch={cfg["batch"]}, epochs={cfg["epochs"]})\n{"="*60}')
    _free_gpu()
    model = YOLO(f'{variant}.pt')
    n_params = sum(p.numel() for p in model.model.parameters())
    t0 = time.time()
    train_res = model.train(name=key, **cfg)
    train_time = time.time() - t0
    val = model.val(data=DATA_YAML, split='val', verbose=False)
    record = dict(
        mAP50    = float(val.box.map50),
        mAP50_95 = float(val.box.map),
        precision= float(val.box.mp),
        recall   = float(val.box.mr),
        params   = n_params,
        train_time_s = train_time,
        best_weights = str(train_res.save_dir / 'weights' / 'best.pt'),
    )
    if store: results_summary[key] = record
    print(f'\n{key}: mAP50={val.box.map50:.4f}  mAP50-95={val.box.map:.4f}  P={val.box.mp:.4f}  R={val.box.mr:.4f}  params={n_params/1e6:.2f}M  time={train_time/60:.1f}min')
    del model, train_res, val
    _free_gpu()
    return record

## Train all 4 variants (n / s / m / l)

Brief explicitly encourages multi-scale comparison. `x` is skipped — diminishing returns and meaningful extra time even on A100.

In [ ]:
for variant in ['yolo26n', 'yolo26s', 'yolo26m', 'yolo26l']:
    train_and_eval(variant)

## Comparison table

In [ ]:
import pandas as pd
df = pd.DataFrame(results_summary).T
df['params_M'] = df['params'] / 1e6
df = df[['mAP50', 'mAP50_95', 'precision', 'recall', 'params_M', 'train_time_s']]
df.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Params (M)', 'Train time (s)']
df = df.round(4)
print(df.to_string())
df.to_csv('/content/yolo26_comparison.csv')
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(df.index, df['mAP@50-95'], color='#4C86E8')
axes[0].set_ylabel('mAP@50-95'); axes[0].set_title('Validation mAP@50-95')
axes[0].set_ylim(0, max(df['mAP@50-95'])*1.15)
for i, v in enumerate(df['mAP@50-95']):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center')
axes[1].scatter(df['Params (M)'], df['mAP@50-95'], s=120, c='#4C86E8')
for name, row in df.iterrows():
    axes[1].annotate(name, (row['Params (M)'], row['mAP@50-95']), xytext=(5, 5), textcoords='offset points')
axes[1].set_xlabel('Parameters (M)'); axes[1].set_ylabel('mAP@50-95'); axes[1].set_title('Accuracy vs Model Size')
plt.tight_layout(); plt.show()

## Per-model diagnostic plots (best variant)

Ultralytics writes confusion matrix, PR curve, training curves, and val-batch previews automatically. Quote the best variant's plots in the write-up.

In [ ]:
from IPython.display import Image as IPyImage, display

BEST_VARIANT = df['mAP@50-95'].idxmax()
print(f'Best variant by mAP@50-95: {BEST_VARIANT}')
RUN_DIR = pathlib.Path(f'/content/runs/detect/{BEST_VARIANT}')
for fname in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'val_batch0_pred.jpg']:
    p = RUN_DIR / fname
    if p.exists():
        print(fname); display(IPyImage(filename=str(p), width=700))

## Test-set evaluation (standard + TTA)

Reports test metrics for the best variant. Also runs **Test-Time Augmentation** (`augment=True`) which does horizontal flip and multi-scale inference, then merges predictions — typically +1–2 mAP50-95 for free. Worth mentioning in the write-up as a deployment optimisation.

In [ ]:
def evaluate_test(variant):
    """Run test-set eval (standard + TTA) for `variant`, save CSV, return the loaded model.
    Defined as a function so it can be re-called after tuning if the final winner differs."""
    best_weights = results_summary[variant]['best_weights']
    model = YOLO(best_weights)

    # Standard
    tr = model.val(data=DATA_YAML, split='test', verbose=False)
    print(f'TEST SET ({variant}, no TTA):')
    print(f'  mAP50    = {tr.box.map50:.4f}')
    print(f'  mAP50-95 = {tr.box.map:.4f}')
    print(f'  Precision= {tr.box.mp:.4f}')
    print(f'  Recall   = {tr.box.mr:.4f}')

    # With TTA
    tta = model.val(data=DATA_YAML, split='test', augment=True, verbose=False)
    print(f'\nTEST SET ({variant}, with TTA):')
    print(f'  mAP50    = {tta.box.map50:.4f}')
    print(f'  mAP50-95 = {tta.box.map:.4f}')
    print(f'  Precision= {tta.box.mp:.4f}')
    print(f'  Recall   = {tta.box.mr:.4f}')

    print(f'\nTTA delta:')
    print(f'  mAP50    {tta.box.map50 - tr.box.map50:+.4f}')
    print(f'  mAP50-95 {tta.box.map  - tr.box.map :+.4f}')

    test_df = pd.DataFrame({
        'no_TTA':   [tr.box.map50,  tr.box.map,  tr.box.mp,  tr.box.mr],
        'with_TTA': [tta.box.map50, tta.box.map, tta.box.mp, tta.box.mr],
    }, index=['mAP50', 'mAP50-95', 'Precision', 'Recall']).round(4)
    test_df.to_csv('/content/yolo26_test_metrics.csv')
    print(test_df)
    return model

best_model = evaluate_test(BEST_VARIANT)

## Test-set prediction gallery

In [ ]:
import random
from PIL import Image
import supervision as sv

test_imgs = sorted((DATA_DIR / 'test' / 'images').iterdir())
sample = random.sample(test_imgs, k=min(9, len(test_imgs)))
box_ann = sv.BoxAnnotator(); label_ann = sv.LabelAnnotator(text_scale=0.5)
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, p in zip(axes.flat, sample):
    img = Image.open(p)
    res = best_model.predict(img, verbose=False)[0]
    det = sv.Detections.from_ultralytics(res)
    labels = [f'pool {c:.2f}' for c in det.confidence]
    out = box_ann.annotate(scene=img.copy(), detections=det)
    out = label_ann.annotate(scene=out, detections=det, labels=labels)
    ax.imshow(out); ax.set_title(p.name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## Failure case extraction

Saves false positives (predicted box with no GT match) and false negatives (GT with no prediction match) from the test set. Feeds the mandatory 5-failure-case write-up section. Matching rule: IoU > 0.5.

**Color key in saved images:**
- Green box: true positive (matched GT)
- Red box: false positive (no GT match)
- Yellow box: false negative (missed GT)

In [ ]:
import cv2

FAILURE_DIR = pathlib.Path('/content/failure_cases')

def iou_xywh(a, b):
    ax1, ay1 = a[0]-a[2]/2, a[1]-a[3]/2
    ax2, ay2 = a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1 = b[0]-b[2]/2, b[1]-b[3]/2
    bx2, by2 = b[0]+b[2]/2, b[1]+b[3]/2
    iw = max(0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter / union if union > 0 else 0.0

def load_gt(lbl_path):
    """Return list of [xc, yc, w, h] bboxes from a YOLO label file (handles both bbox and polygon formats)."""
    if not lbl_path.exists(): return []
    out = []
    for line in open(lbl_path):
        parts = line.split()
        if len(parts) < 5: continue
        coords = list(map(float, parts[1:]))
        if len(coords) == 4:
            out.append(coords)
        else:
            xs = coords[0::2]; ys = coords[1::2]
            out.append([(min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)])
    return out

test_imgs_dir = DATA_DIR / 'test' / 'images'
test_lbls_dir = DATA_DIR / 'test' / 'labels'

def extract_failures(model, label='best'):
    """Dump TP/FP/FN annotated test images and per-bucket counts. Overwrites FAILURE_DIR."""
    if FAILURE_DIR.exists(): shutil.rmtree(FAILURE_DIR)
    (FAILURE_DIR / 'fp').mkdir(parents=True)
    (FAILURE_DIR / 'fn').mkdir(parents=True)
    (FAILURE_DIR / 'mixed').mkdir(parents=True)
    counts = dict(clean=0, fp=0, fn=0, mixed=0)
    total_fps = total_fns = total_tps = 0

    for img_path in sorted(test_imgs_dir.iterdir()):
        gts = load_gt(test_lbls_dir / (img_path.stem + '.txt'))
        res = model.predict(str(img_path), conf=0.25, verbose=False)[0]
        img = res.orig_img.copy()
        H, W = img.shape[:2]
        preds = []
        for box in res.boxes:
            xc, yc, w, h = box.xywhn[0].tolist()
            conf = float(box.conf[0])
            preds.append([xc, yc, w, h, conf])
        matched_gt = set(); matched_pred = set()
        for i, p in enumerate(preds):
            best_j, best_iou = -1, 0
            for j, g in enumerate(gts):
                if j in matched_gt: continue
                iou = iou_xywh(p[:4], g)
                if iou > 0.5 and iou > best_iou:
                    best_j, best_iou = j, iou
            if best_j >= 0:
                matched_gt.add(best_j); matched_pred.add(i)
        fps = [p for i, p in enumerate(preds) if i not in matched_pred]
        fns = [g for j, g in enumerate(gts) if j not in matched_gt]
        total_fps += len(fps); total_fns += len(fns); total_tps += len(matched_pred)
        if not fps and not fns:
            counts['clean'] += 1
            continue
        annotated = img.copy()
        for i, p in enumerate(preds):
            x1, y1 = int((p[0]-p[2]/2)*W), int((p[1]-p[3]/2)*H)
            x2, y2 = int((p[0]+p[2]/2)*W), int((p[1]+p[3]/2)*H)
            color = (0, 255, 0) if i in matched_pred else (0, 0, 255)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
            cv2.putText(annotated, f'{p[4]:.2f}', (x1, max(y1-3, 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        for j, g in enumerate(gts):
            if j in matched_gt: continue
            x1, y1 = int((g[0]-g[2]/2)*W), int((g[1]-g[3]/2)*H)
            x2, y2 = int((g[0]+g[2]/2)*W), int((g[1]+g[3]/2)*H)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 255), 2)
            cv2.putText(annotated, 'MISS', (x1, max(y1-3, 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
        if fps and fns: bucket = 'mixed'
        elif fps:      bucket = 'fp'
        else:          bucket = 'fn'
        counts[bucket] += 1
        cv2.imwrite(str(FAILURE_DIR / bucket / img_path.name), annotated)

    print(f'Test set results ({label}):')
    print(f'  Total TPs: {total_tps}')
    print(f'  Total FPs: {total_fps}')
    print(f'  Total FNs: {total_fns}')
    print(f'\nImage-level buckets ({sum(counts.values())} test images):')
    for k, v in counts.items(): print(f'  {k:8s} = {v}')
    print(f'\nFailure images saved to {FAILURE_DIR}/(fp|fn|mixed)/')
    print('Color key: green=TP, red=FP, yellow=missed GT')

extract_failures(best_model, label=BEST_VARIANT)

## Bounding-box size distribution

Useful for the small-object discussion in the RF-DETR vs YOLO comparison. Plots GT bbox area as a fraction of the image area, with COCO-style small/medium/large bins.

In [ ]:
split_areas = {}
for split in ['train', 'valid', 'test']:
    lbl_dir = DATA_DIR / split / 'labels'
    areas = []
    for lbl in lbl_dir.iterdir():
        for line in open(lbl):
            parts = line.split()
            if len(parts) < 5: continue
            coords = list(map(float, parts[1:]))
            if len(coords) == 4:
                _, _, w, h = coords
            else:
                xs = coords[0::2]; ys = coords[1::2]
                w, h = max(xs)-min(xs), max(ys)-min(ys)
            areas.append(w * h)
    split_areas[split] = areas
    print(f'{split}: {len(areas)} boxes, mean area = {np.mean(areas):.4f}, median = {np.median(areas):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split, areas in split_areas.items():
    axes[0].hist(areas, bins=20, alpha=0.5, label=split, density=True)
axes[0].set_xlabel('Bbox area (fraction of image)'); axes[0].set_ylabel('Density')
axes[0].set_title('Bbox area distribution'); axes[0].legend()

# COCO-style bins (relative to 640x640)
small_pct = [sum(1 for a in areas if a < 0.0025)             / max(1, len(areas)) * 100 for areas in split_areas.values()]
med_pct   = [sum(1 for a in areas if 0.0025 <= a < 0.022)   / max(1, len(areas)) * 100 for areas in split_areas.values()]
lrg_pct   = [sum(1 for a in areas if a >= 0.022)             / max(1, len(areas)) * 100 for areas in split_areas.values()]
x = np.arange(3); width = 0.25
axes[1].bar(x - width, small_pct, width, label='small (<32²)')
axes[1].bar(x,         med_pct,   width, label='medium (<96²)')
axes[1].bar(x + width, lrg_pct,   width, label='large (≥96²)')
axes[1].set_xticks(x); axes[1].set_xticklabels(['train', 'valid', 'test'])
axes[1].set_ylabel('% of boxes'); axes[1].set_title('COCO-style size bins')
axes[1].legend()
plt.tight_layout(); plt.show()

## Augmentation ablation study

Trains yolo26n with 5 progressively-richer augmentation configurations on the same data, same other hyperparameters. Directly answers brief question 6: "Which augmentations improved performance the most?"

Each run: 60 epochs (shorter to stay in budget), patience=20, batch=16. Configs:
- **none**: all augmentations disabled
- **mosaic_only**: just mosaic stitching
- **mosaic_hsv**: + HSV color jitter
- **mosaic_hsv_geo**: + flips (lr+ud) + 180° rotation + translate/scale (aerial-specific)
- **full**: + mixup (matches the main runs)

Time budget: ~50 minutes on T4 / ~12-15 minutes on A100.

In [ ]:
ABLATION_CONFIGS = {
    'none':           dict(mosaic=0.0, mixup=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
                            fliplr=0.0, flipud=0.0, degrees=0.0, translate=0.0, scale=0.0),
    'mosaic_only':    dict(mosaic=1.0, mixup=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
                            fliplr=0.0, flipud=0.0, degrees=0.0, translate=0.0, scale=0.0),
    'mosaic_hsv':     dict(mosaic=1.0, mixup=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                            fliplr=0.0, flipud=0.0, degrees=0.0, translate=0.0, scale=0.0),
    'mosaic_hsv_geo': dict(mosaic=1.0, mixup=0.0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                            fliplr=0.5, flipud=0.5, degrees=180, translate=0.1, scale=0.5),
    'full':           dict(mosaic=1.0, mixup=0.05, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                            fliplr=0.5, flipud=0.5, degrees=180, translate=0.1, scale=0.5),
}

ablation_results = {}
for ab_name, ab_overrides in ABLATION_CONFIGS.items():
    cfg = dict(BASE_CFG)
    cfg['batch'] = VARIANT_BATCH['yolo26n']
    cfg['epochs'] = 60
    cfg['patience'] = 20
    cfg.update(ab_overrides)
    # close_mosaic only makes sense if mosaic is on and < epochs
    cfg['close_mosaic'] = 10 if cfg['mosaic'] > 0 else 0
    rec = train_and_eval('yolo26n', cfg=cfg, name_suffix=f'_ab_{ab_name}', store=False)
    ablation_results[ab_name] = rec

df_ab = pd.DataFrame(ablation_results).T
df_ab = df_ab[['mAP50', 'mAP50_95', 'precision', 'recall', 'train_time_s']].astype(float).round(4)
df_ab.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Train time (s)']
print(df_ab.to_string())
df_ab.to_csv('/content/yolo26_ablation.csv')

fig, ax = plt.subplots(figsize=(10, 5))
configs = list(ablation_results.keys())
x = np.arange(len(configs)); width = 0.35
ax.bar(x - width/2, [ablation_results[c]['mAP50']    for c in configs], width, label='mAP@50')
ax.bar(x + width/2, [ablation_results[c]['mAP50_95'] for c in configs], width, label='mAP@50-95')
ax.set_xticks(x); ax.set_xticklabels(configs, rotation=15)
ax.set_ylabel('Validation mAP'); ax.set_title('Augmentation ablation (yolo26n × 60 epochs)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i, c in enumerate(configs):
    ax.text(i - width/2, ablation_results[c]['mAP50']+0.005, f'{ablation_results[c]["mAP50"]:.3f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## Hyperparameter tuning (Ultralytics genetic-algorithm tuner)

Runs `model.tune()` on yolo26n for **10 iterations × 30 epochs each**. Ultralytics' tuner uses mutation-based search over learning rates, momentum, weight decay, warmup, loss weights, and augmentation strengths.

After tuning, retrains yolo26n with the best-found hyperparameters at full epoch count for the final number.

Time budget: ~1.5 hr on T4 / ~20-25 minutes on A100 (10 iterations × ~2 min each + final retrain ~3-5 min).

In [ ]:
# Ultralytics' built-in tuner (mutation-based GA, doesn't require external Ray Tune)
tune_model = YOLO('yolo26n.pt')
tune_model.tune(
    data=DATA_YAML,
    epochs=30,
    iterations=10,
    optimizer='AdamW',
    plots=False,
    save=False,
    val=True,
    name='tune',
    imgsz=640,
    batch=VARIANT_BATCH['yolo26n'],
    cache='ram',
    seed=0,
    deterministic=True,
    patience=15,
)
del tune_model; _free_gpu()

In [ ]:
# Read best HPs and retrain yolo26n with them.
# Pick the most-recently-modified tune* dir so re-runs grab the latest run, not a stale one.
tune_candidates = sorted(
    [c for c in pathlib.Path('/content/runs/detect').iterdir()
     if c.name.startswith('tune') and (c / 'best_hyperparameters.yaml').exists()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
best_hp_path = (tune_candidates[0] / 'best_hyperparameters.yaml') if tune_candidates else None
if tune_candidates:
    print(f'Found {len(tune_candidates)} tune dir(s); using latest by mtime: {tune_candidates[0].name}')

if best_hp_path is not None:
    with open(best_hp_path) as f:
        best_hp = yaml.safe_load(f)
    print(f'Best HPs from {best_hp_path}:')
    for k, v in best_hp.items(): print(f'  {k:18s} = {v}')

    tuned_cfg = dict(BASE_CFG)
    tuned_cfg['batch'] = VARIANT_BATCH['yolo26n']
    tuned_cfg['optimizer'] = 'AdamW'  # explicit: tuner used AdamW; with optimizer='auto' the tuned lr0/momentum are silently overridden
    # Only override known hyperparams (not all keys in best_hp may be valid for train())
    valid_keys = {'lr0','lrf','momentum','weight_decay','warmup_epochs','warmup_momentum',
                  'box','cls','dfl','hsv_h','hsv_s','hsv_v','degrees','translate','scale',
                  'shear','perspective','flipud','fliplr','bgr','mosaic','mixup','copy_paste',
                  'close_mosaic'}
    for k, v in best_hp.items():
        if k in valid_keys:
            tuned_cfg[k] = v
    print(f'\nRetraining yolo26n with tuned HPs at {BASE_CFG["epochs"]} epochs...')
    train_and_eval('yolo26n', cfg=tuned_cfg, name_suffix='_tuned', store=True)
else:
    print('No best_hyperparameters.yaml found — check /content/runs/detect/tune* output dir')

## Updated comparison table (with tuned variant)

Re-emits the comparison table including the tuned yolo26n if the tuning run completed.

In [ ]:
df = pd.DataFrame(results_summary).T
df['params_M'] = df['params'] / 1e6
df = df[['mAP50', 'mAP50_95', 'precision', 'recall', 'params_M', 'train_time_s']]
df.columns = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'Params (M)', 'Train time (s)']
df = df.round(4)
print(df.to_string())
df.to_csv('/content/yolo26_comparison.csv')

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df.index, df['mAP@50-95'], color=['#4C86E8']*4 + ['#E84C4C']*(len(df)-4))
ax.set_ylabel('mAP@50-95'); ax.set_title('Validation mAP@50-95 — all variants + tuned')
plt.xticks(rotation=15)
for i, v in enumerate(df['mAP@50-95']):
    ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=8)
plt.tight_layout(); plt.show()
df

## Post-tuning re-evaluation (ensure CSV + failures match the final winner)

The test-eval cell and the failure-extraction cell above ran with the best of the n/s/m/l set (`BEST_VARIANT`), because they're placed before the tuning section. If the tuning section produces a different winner (typically `yolo26n_tuned` wins on mAP@50-95), we need to re-run test eval and failure extraction so `yolo26_test_metrics.csv` and `/content/failure_cases/` describe the **final** best model, not the pre-tuning one. The cell below does that automatically.

In [ ]:
FINAL_BEST = df['mAP@50-95'].idxmax()
print(f'Initial best (pre-tuning):  {BEST_VARIANT}')
print(f'Final best (post-tuning):   {FINAL_BEST}')

if FINAL_BEST == BEST_VARIANT:
    print('\nFinal winner matches initial winner — yolo26_test_metrics.csv and /content/failure_cases/ are already accurate.')
else:
    print(f'\nFinal winner differs — re-running test eval + failure extraction for {FINAL_BEST}.')
    print('(Overwrites yolo26_test_metrics.csv and /content/failure_cases/.)\n')
    best_model = evaluate_test(FINAL_BEST)
    extract_failures(best_model, label=FINAL_BEST)
    BEST_VARIANT = FINAL_BEST   # downstream cells (gallery, Drive bundle) now refer to the final winner

## Polygon → OBB label conversion (Step 4 enabler)

The Roboflow export contains polygon annotations (from your manual review pass). For Step 4 (oriented bounding boxes), we derive the minimum-area rotated bounding box from each polygon using `cv2.minAreaRect()`. **No re-annotation needed.**

Output: `/content/pool_dataset_obb/` mirrors the YOLO directory layout. Label files are YOLO-OBB format: `class x1 y1 x2 y2 x3 y3 x4 y4`, normalised to image dimensions, 4 rotated-rectangle corners in clockwise order.

For images that have axis-aligned bboxes (5-token format — a handful in the dataset), we expand to the 4 corner format with no rotation.

In [ ]:
OBB_DIR = pathlib.Path('/content/pool_dataset_obb')
if OBB_DIR.exists(): shutil.rmtree(OBB_DIR)

def polygon_to_obb(coords, img_w, img_h):
    pts = np.array(coords).reshape(-1, 2)
    pts_px = (pts * np.array([img_w, img_h])).astype(np.float32)
    rect = cv2.minAreaRect(pts_px)
    box = cv2.boxPoints(rect)
    return (box / np.array([img_w, img_h])).flatten().tolist()

def bbox_to_obb(xc, yc, w, h):
    return [xc-w/2, yc-h/2,  xc+w/2, yc-h/2,  xc+w/2, yc+h/2,  xc-w/2, yc+h/2]

stats = dict(polygon=0, bbox=0, empty=0)
for split in ['train', 'valid', 'test']:
    src_img_dir = DATA_DIR / split / 'images'
    src_lbl_dir = DATA_DIR / split / 'labels'
    dst_img_dir = OBB_DIR / split / 'images'
    dst_lbl_dir = OBB_DIR / split / 'labels'
    dst_img_dir.mkdir(parents=True); dst_lbl_dir.mkdir(parents=True)
    for img_path in src_img_dir.iterdir():
        shutil.copy(img_path, dst_img_dir / img_path.name)
        lbl_path = src_lbl_dir / (img_path.stem + '.txt')
        out_lines = []
        if lbl_path.exists():
            img_w, img_h = Image.open(img_path).size
            for line in open(lbl_path):
                parts = line.split()
                if len(parts) < 5:
                    stats['empty'] += 1; continue
                cls = parts[0]
                coords = list(map(float, parts[1:]))
                if len(coords) == 4:
                    obb = bbox_to_obb(*coords); stats['bbox'] += 1
                else:
                    obb = polygon_to_obb(coords, img_w, img_h); stats['polygon'] += 1
                obb = [max(0.0, min(1.0, v)) for v in obb]
                out_lines.append(f'{cls} ' + ' '.join(f'{v:.6f}' for v in obb))
        with open(dst_lbl_dir / (img_path.stem + '.txt'), 'w') as f:
            f.write('\n'.join(out_lines))

# OBB data.yaml
obb_yaml = dict(
    train=str(OBB_DIR / 'train' / 'images'),
    val=str(OBB_DIR / 'valid' / 'images'),
    test=str(OBB_DIR / 'test' / 'images'),
    nc=1,
    names=['swimming pool'],
)
with open(OBB_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump(obb_yaml, f)

print(f'Converted: {stats["polygon"]} polygons → OBB, {stats["bbox"]} axis-aligned bboxes → 4-corner format, {stats["empty"]} empty lines skipped')

### OBB sanity check

In [ ]:
# Visualise OBB on a handful of training images
train_obb_imgs = list((OBB_DIR / 'train' / 'images').iterdir())
sample = random.sample(train_obb_imgs, k=min(6, len(train_obb_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample):
    img = np.array(Image.open(img_path))
    H, W = img.shape[:2]
    ax.imshow(img)
    lbl = OBB_DIR / 'train' / 'labels' / (img_path.stem + '.txt')
    if lbl.exists():
        for line in open(lbl):
            parts = line.split()
            if len(parts) < 9: continue
            coords = list(map(float, parts[1:]))
            pts = np.array(coords).reshape(-1, 2) * np.array([W, H])
            pts = np.vstack([pts, pts[0]])
            ax.plot(pts[:, 0], pts[:, 1], 'lime', linewidth=2)
    ax.set_title(img_path.stem[:25], fontsize=8); ax.axis('off')
plt.suptitle('OBB sanity check (green = derived rotated bbox)')
plt.tight_layout(); plt.show()

## Bundle artifacts to Drive

Saves training runs, comparison CSVs, failure cases, and the OBB dataset for Step 4.

In [ ]:
DRIVE_OUT = '/content/drive/MyDrive/IE/IndividualAssignmentMBD2026'

# Training runs (curves, weights, val previews)
shutil.make_archive('/content/yolo26_runs', 'zip', '/content/runs')
shutil.copy('/content/yolo26_runs.zip', f'{DRIVE_OUT}/yolo26_runs.zip')

# Comparison tables
for f in ['yolo26_comparison.csv', 'yolo26_ablation.csv', 'yolo26_test_metrics.csv']:
    src = f'/content/{f}'
    if pathlib.Path(src).exists():
        shutil.copy(src, f'{DRIVE_OUT}/{f}')

# Failure cases for write-up
shutil.make_archive('/content/yolo26_failures', 'zip', '/content/failure_cases')
shutil.copy('/content/yolo26_failures.zip', f'{DRIVE_OUT}/yolo26_failures.zip')

# OBB dataset for Step 4
shutil.make_archive('/content/pool_dataset_obb', 'zip', '/content/pool_dataset_obb')
shutil.copy('/content/pool_dataset_obb.zip', f'{DRIVE_OUT}/pool_dataset_obb.zip')

print('Saved to Drive:')
print('  yolo26_runs.zip          — all training output dirs (curves, weights, val previews)')
print('  yolo26_comparison.csv    — n/s/m/l + tuned variant comparison')
print('  yolo26_ablation.csv      — augmentation ablation results')
print('  yolo26_test_metrics.csv  — best variant test metrics (with/without TTA)')
print('  yolo26_failures.zip      — false positive/negative test images for write-up')
print('  pool_dataset_obb.zip     — converted OBB dataset for Step 4')

## Hardware used (for the write-up)

Auto-detected in the first cell. **This run targets A100 (40 GB, Ampere, Pro+ tier).** For reference:
- T4: 16 GB VRAM, 2,560 CUDA cores, Turing architecture (Colab free / Pro)
- L4: 24 GB, 7,680 CUDA cores, Ada Lovelace
- A100: 40 GB, Ampere (Pro+ tier) ← used here

Training time figures in the comparison table are wall-clock seconds on the detected GPU.